In [1]:
"hi"

'hi'

In [32]:
import ast

code = """
y = x * 2
Print(y)"""
tree = ast.parse(code)

print(ast.dump(tree, indent=4))

Module(
    body=[
        Assign(
            targets=[
                Name(id='y', ctx=Store())],
            value=BinOp(
                left=Name(id='x', ctx=Load()),
                op=Mult(),
                right=Constant(value=2))),
        Expr(
            value=Call(
                func=Name(id='Print', ctx=Load()),
                args=[
                    Name(id='y', ctx=Load())]))])


In [3]:
for node in tree.body:
    print(type(node))

<class 'ast.Assign'>
<class 'ast.Assign'>


In [33]:
for node in ast.walk(tree):
    print(type(node).__name__)

Module
Assign
Expr
Name
BinOp
Call
Store
Name
Mult
Constant
Name
Name
Load
Load
Load


In [8]:
class Visitor(ast.NodeVisitor):
    def visit(self, node):
        print(type(node).__name__)
        self.generic_visit(node)

tree = ast.parse("x = a + b")
Visitor().visit(tree)

Module
Assign
Name
Store
BinOp
Name
Load
Add
Name
Load


In [7]:
class Transformer(ast.NodeTransformer):
    def visit_Call(self, node):
        if isinstance(node.func, ast.Name) and node.func.id == "print":
            node.func.id = "logger.info"
        return self.generic_visit(node)

In [11]:
tree = ast.parse("x = a + 2*b")
new_tree = Transformer().visit(tree)
ast.fix_missing_locations(new_tree)

In [18]:
import ast

class UsageVisitor(ast.NodeVisitor):
    def __init__(self):
        self.assigned = set()
        self.used = set()
    
    def visit_Name(self, node):
        if isinstance(node.ctx, ast.Store):
            self.assigned.add(node.id)
        elif isinstance(node.ctx, ast.Load):
            self.used.add(node.id)

code = """
x = 10
y = 20
print(y)
"""

tree = ast.parse(code)
visitor = UsageVisitor()
visitor.visit(tree)
print("Assigned:", visitor.assigned)
print("Used:", visitor.used)
print("Unused:", visitor.assigned - visitor.used)

Assigned: {'x', 'y'}
Used: {'print', 'y'}
Unused: {'x'}


In [ ]:
class Scope:
    def __init__(self):
        self.assigned = set()
        self.used = set()



class ScopedVisitor(ast.NodeVisitor):
    def __init__(self):
        self.scopes = [Scope()]  # global scope

    def current_scope(self):
        return self.scopes[-1]

    def visit_FunctionDef(self, node):
        self.scopes.append(Scope())
        self.generic_visit(node)
        scope = self.scopes.pop()

        unused = scope.assigned - scope.used
        print(f"In function {node.name}, unused:", unused)

    def visit_Name(self, node):
        if isinstance(node.ctx, ast.Store):
            self.current_scope().assigned.add(node.id)
        elif isinstance(node.ctx, ast.Load):
            for scope in reversed(self.scopes):
                if node.id in scope.assigned:
                    scope.used.add(node.id)
                    break




In [29]:
code = """
x = 10
y=20

def foo():
    y = 20
    print(y)

def bar():
    z = 30
"""

tree = ast.parse(code)
visitor = ScopedVisitor()
visitor.visit(tree)

print("Assigned:", visitor.scopes[0].assigned)
print("Used:", visitor.scopes[0].used)
print("Unused:", visitor.scopes[0].assigned - visitor.scopes[0].used)
print(visitor.scopes[0])



In function foo, unused: set()
In function bar, unused: {'z'}
Assigned: {'x', 'y'}
Used: set()
Unused: {'x', 'y'}


In [30]:
import ast

class PrintToLogger(ast.NodeTransformer):
    def visit_Call(self, node):
        # First transform children
        self.generic_visit(node)

        # Match print(...)
        if isinstance(node.func, ast.Name) and node.func.id == "print":
            node.func = ast.Attribute(
                value=ast.Name(id="logger", ctx=ast.Load()),
                attr="info",
                ctx=ast.Load()
            )
        return node

In [31]:
code = """
def foo():
    print("hello")
"""

tree = ast.parse(code)
new_tree = PrintToLogger().visit(tree)
ast.fix_missing_locations(new_tree)

print(ast.unparse(new_tree))

def foo():
    logger.info('hello')


In [34]:
class Block:
    def __init__(self, name):
        self.name = name
        self.statements = []
        self.next_blocks = []

    def __repr__(self):
        return f"<Block {self.name}>"

In [ ]:
import ast

class CFGBuilder(ast.NodeVisitor):
    def __init__(self):
        self.blocks = []
        self.current = self.new_block("entry")

    def new_block(self, name):
        block = Block(name)
        self.blocks.append(block)
        return block

    def visit_Assign(self, node):
        if self.current is not None:
            self.current.statements.append(node)

    def visit_Expr(self, node):
        if self.current is not None:
            self.current.statements.append(node)

    def visit_Return(self, node):
        if self.current is not None:
            self.current.statements.append(node)
            # return TERMINATES flow
            self.current.next_blocks = []
            self.current = None

    def visit_If(self, node):
        cond_block = self.current

        then_block = self.new_block("then")
        else_block = self.new_block("else")
        after_block = self.new_block("after")

        cond_block.next_blocks = [then_block, else_block]

        # then branch
        self.current = then_block
        for stmt in node.body:
            self.visit(stmt)
        if self.current is not None:
            self.current.next_blocks.append(after_block)

        # else branch
        self.current = else_block
        for stmt in node.orelse:
            self.visit(stmt)
        if self.current is not None:
            self.current.next_blocks.append(after_block)

        self.current = after_block

In [36]:
code = """
def f(x):
    if x > 0:
        return 1
    print("alive")
    return 2
    print("dead")
"""

tree = ast.parse(code)
builder = CFGBuilder()
builder.visit(tree)
blocks = builder.blocks

AttributeError: 'NoneType' object has no attribute 'statements'